# Peak Popularity Prediction

Does adding early review features improve peak-popularity prediction over a time-series model alone?

Three models are compared with walk-forward cross-validation:
- Baseline: historical mean peak per style (no early-season data)
- Model A: RidgeCV on style + time-series features
- Model B: RidgeCV on style + time-series + review features


In [2]:
import numpy as np
import pandas as pd
from sklearn.linear_model import RidgeCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from pathlib import Path

RESULTS_DIR = Path('results')

TS_COLS  = ['ts_mean', 'ts_slope', 'ts_curvature', 'ts_range', 'ts_peak_pos']
REV_COLS = ['rev_count', 'rev_mean_rating', 'rev_positive_frac']
TARGET   = 'actual_peak_value'
ALPHAS   = [0.01, 0.1, 1.0, 10.0, 100.0]

def mae(actual, predicted):
    """Mean Absolute Error between two array-like sequences."""
    return float(np.mean(np.abs(np.array(actual, dtype=float) - np.array(predicted, dtype=float))))

def make_pipe(num_cols):
    """Ridge regression pipeline: OneHotEncoder for style + StandardScaler + RidgeCV."""
    ct = ColumnTransformer([
        ('ohe', OneHotEncoder(handle_unknown='ignore'), ['style']),
        ('sc',  StandardScaler(), num_cols),
    ])
    return Pipeline([('ct', ct), ('ridge', RidgeCV(alphas=ALPHAS))])

def make_rf_pipe(num_cols):
    """Random Forest pipeline: OneHotEncoder for style, no scaling (trees are scale-invariant)."""
    ct = ColumnTransformer([
        ('ohe', OneHotEncoder(handle_unknown='ignore'), ['style']),
        ('pass', 'passthrough', num_cols),
    ])
    return Pipeline([('ct', ct), ('rf', RandomForestRegressor(
        n_estimators=200, max_depth=4, random_state=42))])

## Model design

### Ridge regression - mathematics

The closed-form solution for Ridge (L2-regularised least squares) is:

$$\hat{\beta} = (X^\top X + \alpha I)^{-1} X^\top y$$

where $\alpha \geq 0$ is the regularisation strength. As $\alpha \to 0$ this reduces to ordinary OLS; as $\alpha \to \infty$ all coefficients shrink toward zero. RidgeCV selects the best $\alpha$ automatically via leave-one-out cross-validation across the candidate grid [0.01, 0.1, 1, 10, 100].

Why Ridge over OLS? With only 16-40 training instances per fold and ~13 features (8 style dummies + 5 ts features), ordinary least squares would overfit. Ridge's shrinkage improves out-of-fold generalisation at minimal cost to bias.

### Random Forest - comparison model

Random Forest (Model C) is a non-parametric ensemble that makes no linearity or normality assumptions. It is included as a structural test: if Ridge and RF give similar MAE, the linear approximation is appropriate. If RF clearly wins, the true relationship is non-linear and Ridge is mis-specified.

Trees are scale-invariant, so no StandardScaler is applied to the numerical features in make_rf_pipe.

### Style one-hot encoding

Including style as a categorical feature lets each model learn a per-style baseline (how each style typically performs) and then adjust up or down based on early-season signals. Without style encoding the model must extract the style effect from the numerical features alone - a much harder task with 48 instances.

In [3]:
df = pd.read_csv(RESULTS_DIR / 'features.csv')
df = df.sort_values(['year', 'style']).reset_index(drop=True)

N_YEARS = int(df['year'].nunique())
print(f'Instances: {len(df)}, Years: {N_YEARS}')
print(f'Walk-forward folds: train 0..N-1, test N  (N = 2..{N_YEARS - 1})')

Instances: 48, Years: 6
Walk-forward folds: train 0..N-1, test N  (N = 2..5)
